Table of Contents:
1. Import libraries & data.
2. Kepler.gl map: Trip flows between stations.  

In [1]:
# Import libraries

import pandas as pd
from keplergl import KeplerGl
import json
from pathlib import Path

/Users/samantha.lisik/miniforge3/envs/citibike310/lib/python3.10/site-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


In [3]:
# Set path
from pathlib import Path


PROJECT_DIR = Path("/Users/samantha.lisik/Documents/citibike")
CSV_PATH = PROJECT_DIR / "reduced_data_to_plot_2pct.csv"
CSV_PATH.exists()

True

In [4]:
# Check columns

df_head = pd.read_csv(CSV_PATH, nrows=5)
df_head.columns

Index(['start_station_name', 'date', 'avgTemp_C'], dtype='object')

In [ ]:
# Rubric aggregation

use_cols_min = ["start_station_name", "end_station_name"]

df_min = pd.read_csv(csv_path, usecols=use_cols_min)

# Create the required "1" column
df_min["trip"] = 1

# Aggregate to the required 3 columns
df_agg = (
    df_min.groupby(
        ["start_station_name", "end_station_name"],
        as_index=False
    )["trip"]
    .sum()
    .rename(columns={"trip": "trips"})
)

df_agg.head()

In [ ]:
# Clean aggregation + coordinates (memory-safe)

use_cols = [
    "start_station_name", "end_station_name",
    "start_lat", "start_lng", "end_lat", "end_lng"
]

CHUNKSIZE = 200_000

pair_counts = {}          # (start_name, end_name) -> trips
start_coords = {}         # start_name -> (lat, lng)
end_coords = {}           # end_name   -> (lat, lng)

for chunk in pd.read_csv(csv_path, usecols=use_cols, chunksize=CHUNKSIZE):
    chunk = chunk.dropna(subset=["start_station_name","end_station_name","start_lat","start_lng","end_lat","end_lng"])

    # reduce tiny coordinate variation
    chunk["start_lat"] = chunk["start_lat"].round(5)
    chunk["start_lng"] = chunk["start_lng"].round(5)
    chunk["end_lat"]   = chunk["end_lat"].round(5)
    chunk["end_lng"]   = chunk["end_lng"].round(5)

    # store one representative coordinate per station name (first seen)
    for s, lat, lng in zip(chunk["start_station_name"], chunk["start_lat"], chunk["start_lng"]):
        start_coords.setdefault(s, (lat, lng))
    for e, lat, lng in zip(chunk["end_station_name"], chunk["end_lat"], chunk["end_lng"]):
        end_coords.setdefault(e, (lat, lng))

    # count trips per (start, end)
    grouped = chunk.groupby(["start_station_name", "end_station_name"]).size()

    for (s, e), n in grouped.items():
        pair_counts[(s, e)] = pair_counts.get((s, e), 0) + int(n)

# build aggregated df
df_trips = (
    pd.DataFrame(
        [(s, e, trips) for (s, e), trips in pair_counts.items()],
        columns=["start_station_name", "end_station_name", "trips"]
    )
)

# add coordinates back
start_df = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in start_coords.items()],
    columns=["start_station_name", "start_lat", "start_lng"]
)
end_df = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in end_coords.items()],
    columns=["end_station_name", "end_lat", "end_lng"]
)

df_trips = df_trips.merge(start_df, on="start_station_name", how="left").merge(end_df, on="end_station_name", how="left")

In [ ]:
# Quick check
df_trips.sort_values("trips", ascending=False).head(10)

In [ ]:
# Drop self-loops
df_trips = df_trips[
    df_trips.start_station_name != df_trips.end_station_name
]

In [ ]:
# Quick check
df_trips.sort_values("trips", ascending=False).head(10)

# Sanity checks & data cleanup explanation

- Aggregated trips by (start_station_name, end_station_name) using chunked reads to stay memory-safe.

- Rounded station coordinates to 5 decimals to reduce GPS jitter and ensure stable joins.

- Verified coordinate validity and locality (NYC bounds, no lat/lng anomalies).

- Ran a top-flows check and confirmed that the highest-volume pairs were initially dominated by self-loops (start = end), which are common bike-share artifacts (dock corrections / very short trips).

- Removed self-loops prior to visualization to avoid zero-length arcs and distorted scaling in Kepler.

- Re-checked top flows after filtering; remaining routes are short, plausible, and symmetric across nearby stations, indicating healthy aggregation.

Result: the dataset is now suitable for Kepler arc/line layers without visual or statistical artifacts.

2. Kepler.gl map: Trip flows between stations.

In [ ]:
# Initialize the map

map_1 = KeplerGl(
    height=650,
    data={"Trips": df_trips}
)
map_1

### Kepler.gl Map Customization

The start and end station point layers were styled using a warm yellow–orange color
with reduced opacity to provide clear spatial context while remaining visually subtle.

Trip connections were visualized using a sequential color palette ranging from purple
to orange, mapped to the number of trips. This palette choice helps emphasize
high-volume routes while maintaining contrast against the dark basemap.

## Filtering for the most common trips in New York City

To identify the most common trips in New York City, I added a filter on the `trips` variable in Kepler.gl and increased the minimum threshold to remove low-frequency routes. This significantly reduced visual clutter and highlighted only the highest-volume station-to-station connections.

After filtering, the remaining routes cluster strongly in **Manhattan**, with particularly dense activity in **Midtown and Downtown Manhattan** and along the **Hudson River waterfront**. These areas appear especially busy, as many high-volume routes connect nearby stations within short distances. This pattern suggests frequent, repeat trips rather than occasional long-distance travel.

The prominence of Manhattan and waterfront-adjacent corridors is consistent with what is known about Citi Bike usage in New York City. These zones combine high station density, major employment centers, transit hubs, and popular recreational areas such as riverfront bike paths. Together, these factors help explain why these station pairs remain visible even after filtering for only the most common trips.

In [ ]:
# Create a config object and save the map

config = map_1.config

In [ ]:
# Export the map as html

map_1.save_to_html(
    file_name="NYC_CitiBike_Trips.html",
    read_only=False,
    config=config
)

In [ ]:
# Save the config as a JSON file

import json

with open("kepler_config.json", "w") as outfile:
    json.dump(config, outfile)